# Week 2 — Return Distribution Analysis

**Regime-Adaptive Multi-Factor Alpha Engine**  
Week 2 Deliverable: Cross-sectional return distribution, fat-tail analysis, skewness by year.

## Contents
1. Load price data from DB (or synthetic fallback)
2. Compute daily + monthly returns using `ReturnCalculator`
3. Build `ReturnMatrix` and apply universe filter
4. Cross-sectional distribution by year (skewness, kurtosis)
5. Fat-tail check: empirical vs normal distribution
6. Universe size over time

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.factors.returns import ReturnCalculator
from src.factors.return_matrix import ReturnMatrix
from src.factors.universe import UniverseFilter
from src.config import AppConfig, UniverseFilterConfig

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2d3148',
    'axes.labelcolor': '#c8cde8',
    'xtick.color': '#8891b5',
    'ytick.color': '#8891b5',
    'text.color': '#c8cde8',
    'grid.color': '#2d3148',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
ACCENT = '#6c8ebf'
ACCENT2 = '#d4a857'
DANGER = '#e06c75'

print('✅ Imports OK')

## 1. Load or Synthesise Price Data

In [ ]:
USE_DB = False  # set True if PostgreSQL is running with Week 1 data

if USE_DB:
    from src.data.db import get_engine
    from sqlalchemy import text
    engine = get_engine()
    with engine.connect() as conn:
        df_raw = pd.read_sql(
            text('SELECT date, ticker, adj_close, volume FROM prices ORDER BY date, ticker'),
            conn
        )
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    prices = df_raw.set_index(['date', 'ticker']).sort_index()
    print(f'Loaded {len(prices):,} rows from DB for {prices.index.get_level_values("ticker").nunique()} tickers')
else:
    # ── Synthetic data (no DB needed) ────────────────────────────────────────
    rng = np.random.default_rng(42)
    tickers = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'NVDA', 'JPM', 'BRK-B', 'JNJ']
    dates = pd.date_range('2010-01-04', '2024-12-31', freq='B')

    frames = []
    for i, ticker in enumerate(tickers):
        drift = rng.uniform(0.0001, 0.0008)
        vol = rng.uniform(0.01, 0.025)
        log_r = rng.normal(drift, vol, len(dates))
        price = 50.0 * np.exp(np.cumsum(log_r))
        volume = rng.integers(500_000, 20_000_000, len(dates)).astype(float)
        # Simulate one stock going through IPO midway (for IPO filter demo)
        if ticker == 'META':
            ipo_idx = int(len(dates) * 0.2)
            price[:ipo_idx] = np.nan
            volume[:ipo_idx] = np.nan

        df = pd.DataFrame({
            'adj_close': price,
            'close': price,
            'open': price * (1 + rng.normal(0, 0.002, len(dates))),
            'high': price * (1 + rng.uniform(0, 0.01, len(dates))),
            'low': price * (1 - rng.uniform(0, 0.01, len(dates))),
            'volume': volume,
            'ticker': ticker,
        }, index=dates)
        df.index.name = 'date'
        frames.append(df.reset_index())

    raw = pd.concat(frames, ignore_index=True).dropna(subset=['adj_close'])
    prices = raw.set_index(['date', 'ticker']).sort_index()
    print(f'Synthetic data: {len(prices):,} rows for {len(tickers)} tickers, {dates.min().date()} → {dates.max().date()}')

## 2. Compute Returns

In [ ]:
calc = ReturnCalculator()
daily_returns = calc.compute_daily_log_returns(prices)
monthly_returns = calc.compute_monthly_simple_returns(prices)

print(f'Daily return rows:   {len(daily_returns):,}')
print(f'Monthly return rows: {len(monthly_returns):,}')

# Validate
passed, msg = calc.validate_against_index(daily_returns)
status = '✅' if passed else '⚠️'
print(f'{status} Validation: {msg}')

## 3. Build ReturnMatrix & Apply Universe Filter

In [ ]:
# Build wide monthly return matrix
monthly_reset = monthly_returns.reset_index()
rm = ReturnMatrix.from_long(monthly_reset, return_col='simple_return')
print(rm)

# Apply universe filter
uf = UniverseFilter()
universe_cfg = UniverseFilterConfig(
    min_market_cap_bn=0.001,    # relaxed for synthetic data
    min_dollar_vol_percentile=25,
    ipo_exclusion_months=6,
    dollar_vol_lookback_days=60,
)
membership = uf.compute_membership_fast(prices, universe_cfg)

# Resample membership to month-end to align with monthly return matrix
membership['date'] = pd.to_datetime(membership['date'])
membership_monthly = membership.copy()
membership_monthly['date'] = membership_monthly['date'].dt.to_period('M').dt.to_timestamp('M')
membership_monthly = membership_monthly.groupby(['date', 'ticker'])['in_universe'].any().reset_index()

rm_filtered = rm.trim_by_universe(membership_monthly)
print(f'Universe-filtered matrix: {rm_filtered.shape}')

## 4. Annual Cross-Sectional Distribution (Skewness & Kurtosis)

In [ ]:
annual = rm_filtered.annual_stats()
print(annual.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Sectional Return Distribution by Year', fontsize=14, fontweight='bold', color='#e0e4f5')

years = annual.index.tolist()
colors_skew = [DANGER if v < 0 else ACCENT for v in annual['skewness']]
colors_kurt = [DANGER if v > 1 else ACCENT for v in annual['excess_kurtosis']]

# Skewness
ax1 = axes[0]
ax1.bar(years, annual['skewness'], color=colors_skew, alpha=0.85, width=0.7)
ax1.axhline(0, color='#8891b5', linewidth=0.8, linestyle='--')
ax1.set_title('Cross-sectional Skewness by Year')
ax1.set_xlabel('Year')
ax1.set_ylabel('Skewness')
ax1.grid(True, axis='y')

# Excess kurtosis
ax2 = axes[1]
ax2.bar(years, annual['excess_kurtosis'], color=colors_kurt, alpha=0.85, width=0.7)
ax2.axhline(0, color='#8891b5', linewidth=0.8, linestyle='--', label='Normal = 0')
ax2.set_title('Excess Kurtosis by Year (>0 = fat-tailed)')
ax2.set_xlabel('Year')
ax2.set_ylabel('Excess Kurtosis')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.savefig('annual_distribution_stats.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved → annual_distribution_stats.png')

## 5. Fat-Tail Check: Empirical vs Normal

In [ ]:
# Pool all returns across the full sample
all_returns = rm_filtered.data.values.flatten()
all_returns = all_returns[~np.isnan(all_returns)]

mu, sigma = np.mean(all_returns), np.std(all_returns, ddof=1)
x = np.linspace(mu - 5*sigma, mu + 5*sigma, 500)
normal_pdf = stats.norm.pdf(x, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fat-Tail Analysis: Empirical vs Normal Distribution', fontsize=14, fontweight='bold', color='#e0e4f5')

# Left: Histogram + normal overlay
ax1 = axes[0]
ax1.hist(all_returns, bins=60, density=True, color=ACCENT, alpha=0.6, label='Empirical')
ax1.plot(x, normal_pdf, color=ACCENT2, linewidth=2, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')
ax1.set_title('Return Distribution (Monthly)')
ax1.set_xlabel('Monthly Return')
ax1.set_ylabel('Density')
ax1.legend()
ax1.grid(True)

# Right: Q-Q plot
ax2 = axes[1]
theoretical_q = np.linspace(0.001, 0.999, 200)
empirical_quantiles = np.quantile(all_returns, theoretical_q)
normal_quantiles = stats.norm.ppf(theoretical_q, mu, sigma)
ax2.scatter(normal_quantiles, empirical_quantiles, color=ACCENT, alpha=0.5, s=10, label='Data')
qq_min = min(normal_quantiles.min(), empirical_quantiles.min())
qq_max = max(normal_quantiles.max(), empirical_quantiles.max())
ax2.plot([qq_min, qq_max], [qq_min, qq_max], color=ACCENT2, linewidth=1.5, linestyle='--', label='Normal line')
ax2.set_title('Q-Q Plot vs Normal')
ax2.set_xlabel('Theoretical Quantiles')
ax2.set_ylabel('Empirical Quantiles')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('fat_tail_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

# Statistical summary
print(f'\nDistribution Summary (pooled monthly returns):')
print(f'  N observations: {len(all_returns):,}')
print(f'  Mean:           {mu:.6f}')
print(f'  Std:            {sigma:.6f}')
print(f'  Skewness:       {stats.skew(all_returns):.4f}  (0 = symmetric)')
print(f'  Excess kurtosis:{stats.kurtosis(all_returns):.4f} (0 = normal, >0 = fat tails)')

# Jarque-Bera normality test
jb_stat, jb_p = stats.jarque_bera(all_returns)
print(f'\nJarque-Bera test: statistic={jb_stat:.2f}, p-value={jb_p:.2e}')
if jb_p < 0.05:
    print('  → Reject normality (p < 0.05): fat tails confirmed ✅')
else:
    print('  → Cannot reject normality')

## 6. Universe Size Over Time

In [ ]:
# Count in-universe stocks per month
membership_monthly['date'] = pd.to_datetime(membership_monthly['date'])
universe_size = (
    membership_monthly[membership_monthly['in_universe']]
    .groupby('date')['ticker']
    .count()
    .rename('in_universe_count')
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(universe_size.index, universe_size.values, alpha=0.3, color=ACCENT)
ax.plot(universe_size.index, universe_size.values, color=ACCENT, linewidth=1.5)
ax.set_title('Universe Size Over Time (In-Universe Ticker Count per Month)')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Stocks in Universe')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True)

plt.tight_layout()
plt.savefig('universe_size_over_time.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print(f'\nUniverse size summary:')
print(f'  Min: {universe_size.min()} stocks  ({universe_size.idxmin().date()})')
print(f'  Max: {universe_size.max()} stocks  ({universe_size.idxmax().date()})')
print(f'  Mean: {universe_size.mean():.1f} stocks')

## 7. Summary Table

The table below shows annual skewness and kurtosis for the cross-sectional return distribution. Fat-tailed years (excess kurtosis > 1) are highlighted — these are years when extreme returns dominate and equal-weighting would be dangerous.

In [ ]:
def style_kurtosis(val):
    if isinstance(val, float) and val > 1.0:
        return 'color: #e06c75; font-weight: bold'
    return ''

def style_skew(val):
    if isinstance(val, float) and val < -0.5:
        return 'color: #e06c75; font-weight: bold'
    return ''

display_df = annual[['n_obs', 'mean', 'std', 'skewness', 'excess_kurtosis']].copy()
display_df['mean'] = display_df['mean'].map('{:.4f}'.format)
display_df['std'] = display_df['std'].map('{:.4f}'.format)
display_df['skewness'] = display_df['skewness'].map('{:.3f}'.format)
display_df['excess_kurtosis'] = display_df['excess_kurtosis'].map('{:.3f}'.format)

display_df.style.set_caption('Annual Cross-Sectional Return Distribution Stats')